# Annual single-pass pairwise agreement statistics

Pool exact sufficient sums and mergeable distribution summaries from the v5 daily cache. No full-resolution ERA5 results are read by this notebook.

In [1]:
from datetime import date
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from liljegren_analysis import (
    histogram_quantile_statistics,
    pooled_sufficient_statistics,
)

In [2]:
repository = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
database_path = repository / "data" / "daily_pairwise_all_implementations_v5.duckdb"
year = 2025
start_date = date(year, 1, 1)
end_date = date(year, 12, 31)

In [3]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    metadata = connection.execute("SELECT * FROM pairwise_schema").df()
    daily = connection.execute(
        "SELECT * FROM daily_pairwise WHERE result_date BETWEEN ? AND ? ORDER BY result_date, quantity, implementation_a, implementation_b",
        [start_date, end_date],
    ).df()
    histogram = connection.execute(
        """
        SELECT quantity, implementation_a, implementation_b, abs_bin_id,
               min(abs_bin_lower_c) AS abs_bin_lower_c,
               min(abs_bin_upper_c) AS abs_bin_upper_c,
               sum(n)::BIGINT AS n,
               fsum(area_weight_sum) AS area_weight_sum
        FROM daily_pairwise_abs_histogram
        WHERE result_date BETWEEN ? AND ?
        GROUP BY quantity, implementation_a, implementation_b, abs_bin_id
        ORDER BY quantity, implementation_a, implementation_b, abs_bin_id
        """,
        [start_date, end_date],
    ).df()
    quantity_status = connection.execute(
        "SELECT * FROM daily_quantity_status WHERE result_date BETWEEN ? AND ?",
        [start_date, end_date],
    ).df()

In [4]:
if metadata.to_dict("records") != [{
    "schema_version": 5,
    "sample_scope": "all-implementations",
    "result_schema_id": "hsb-wbgt-comparison-results-csv-v3",
    "quantity_scope": "joint",
    "quantity_set_version": 1,
    "implementation_set_version": 1,
    "abs_histogram_version": 1,
}]:
    raise ValueError(f"Unexpected pairwise schema metadata:\n{metadata}")
quantities = ("wbgt", "tg", "tnwb")
pair_columns = ["quantity", "implementation_a", "implementation_b"]
expected_dates = (end_date - start_date).days + 1
coverage = daily.groupby(pair_columns)["result_date"].nunique()
if len(coverage) != 63 or not coverage.eq(expected_dates).all():
    raise ValueError(f"Incomplete daily pair coverage:\n{coverage}")

joint_support = daily.groupby(
    ["result_date", "implementation_a", "implementation_b"]
)[["n", "area_weight_sum"]].nunique()
if joint_support.gt(1).any().any():
    raise ValueError(f"Joint quantity populations differ:\n{joint_support}")

quantity_status["valid"] = (
    quantity_status["status"].eq("ok") & quantity_status["value_present"]
)
quantity_coverage = (
    quantity_status
    .assign(valid_n=lambda frame: frame["n"].where(frame["valid"], 0))
    .groupby(["quantity", "implementation"], as_index=False, sort=True)
    .agg(total_n=("n", "sum"), valid_n=("valid_n", "sum"))
)
quantity_coverage["valid_percent"] = (
    100.0 * quantity_coverage["valid_n"] / quantity_coverage["total_n"]
)
display(coverage)
display(quantity_coverage)

quantity  implementation_a  implementation_b
tg        heatstress        heatstress-jl       365
                            heatstress-r        365
                            liljegren-c         365
                            meteor              365
                            pywbgt              365
                                               ... 
wbgt      liljegren-c       pywbgt              365
                            thermofeel          365
          meteor            pywbgt              365
                            thermofeel          365
          pywbgt            thermofeel          365
Name: result_date, Length: 63, dtype: int64

,quantity,implementation,total_n,valid_n,valid_percent
0,tg,heatstress,3078453651,3078453651,100.000000
1,tg,heatstress-jl,3078453651,2602799610,84.548930
2,tg,heatstress-r,3078453651,3078453651,100.000000
3,tg,liljegren-c,3078453651,3065637880,99.583695
4,tg,meteor,3078453651,3078453651,100.000000
5,tg,pywbgt,3078453651,2864775163,93.058902
6,tg,thermofeel,3078453651,3078161915,99.990523
7,tnwb,heatstress,3078453651,3078453651,100.000000
8,tnwb,heatstress-jl,3078453651,2602761957,84.547706
9,tnwb,heatstress-r,3078453651,3078453651,100.000000


In [5]:
records = []
for pair_key, pair in daily.groupby(pair_columns, sort=True):
    quantity, implementation_a, implementation_b = pair_key
    statistics = pooled_sufficient_statistics(
        pair["n"],
        sum_differences=pair["sum_difference_c"],
        sum_squared_differences=pair["sum_squared_difference_c2"],
        sum_absolute_differences=pair["sum_absolute_difference_c"],
        min_differences=pair["min_difference_c"],
        max_differences=pair["max_difference_c"],
        max_absolute_differences=pair["max_absolute_difference_c"],
        area_weight_sums=pair["area_weight_sum"],
        area_weighted_sum_differences=pair["area_weighted_difference_sum_c"],
        area_weighted_sum_squared_differences=pair["area_weighted_squared_difference_sum_c2"],
        area_weighted_sum_absolute_differences=pair["area_weighted_absolute_difference_sum_c"],
        exceedance_counts_0_1=pair["n_absolute_difference_gt_0_1_c"],
        exceedance_counts_0_2=pair["n_absolute_difference_gt_0_2_c"],
        exceedance_counts_0_5=pair["n_absolute_difference_gt_0_5_c"],
        exceedance_area_weights_0_1=pair["area_weight_absolute_difference_gt_0_1_c"],
        exceedance_area_weights_0_2=pair["area_weight_absolute_difference_gt_0_2_c"],
        exceedance_area_weights_0_5=pair["area_weight_absolute_difference_gt_0_5_c"],
    )
    pair_histogram = histogram.loc[
        histogram["quantity"].eq(quantity)
        & histogram["implementation_a"].eq(implementation_a)
        & histogram["implementation_b"].eq(implementation_b)
    ]
    quantiles = histogram_quantile_statistics(
        pair_histogram["abs_bin_id"],
        bin_lowers=pair_histogram["abs_bin_lower_c"],
        bin_uppers=pair_histogram["abs_bin_upper_c"],
        counts=pair_histogram["n"],
        area_weight_sums=pair_histogram["area_weight_sum"],
        expected_count=statistics["n"],
        expected_area_weight_sum=statistics["area_weight_sum"],
    )
    records.append({
        "quantity": quantity,
        "implementation_a": implementation_a,
        "implementation_b": implementation_b,
        **statistics,
        **quantiles,
    })

yearly_pairwise = pd.DataFrame.from_records(records)

yearly_pairwise

,quantity,implementation_a,implementation_b,n,mean_difference_c,variance_difference,stddev_difference_c,mean_absolute_difference_c,rmse_c,min_difference_c,...,p95_absolute_difference_upper_c,area_weighted_p95_absolute_difference_c,area_weighted_p95_absolute_difference_lower_c,area_weighted_p95_absolute_difference_upper_c,p99_absolute_difference_c,p99_absolute_difference_lower_c,p99_absolute_difference_upper_c,area_weighted_p99_absolute_difference_c,area_weighted_p99_absolute_difference_lower_c,area_weighted_p99_absolute_difference_upper_c
0,tg,heatstress,heatstress-jl,2541112557,0.288130,9.107506,3.017864,1.679458,3.031588,-49.234433,...,7.50,7.375,7.25,7.50,10.375,10.25,10.50,10.625,10.50,10.75
1,tg,heatstress,heatstress-r,2541112557,0.037510,10.367863,3.219917,1.790783,3.220135,-160.860237,...,7.75,7.625,7.50,7.75,10.875,10.75,11.00,11.125,11.00,11.25
2,tg,heatstress,liljegren-c,2541112557,0.259983,9.885031,3.144047,1.740578,3.154778,-38.263651,...,8.00,8.125,8.00,8.25,10.625,10.50,10.75,10.875,10.75,11.00
3,tg,heatstress,meteor,2541112557,1.254350,6.388036,2.527456,1.269426,2.821601,-11.999958,...,8.00,7.875,7.75,8.00,10.125,10.00,10.25,10.125,10.00,10.25
4,tg,heatstress,pywbgt,2541112557,0.297907,8.683419,2.946764,1.640838,2.961785,-19.849101,...,7.75,7.625,7.50,7.75,9.875,9.75,10.00,9.875,9.75,10.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,wbgt,liljegren-c,pywbgt,2541112557,0.025329,0.155823,0.394744,0.086424,0.395555,-2.678223,...,0.36,0.405,0.40,0.41,1.745,1.74,1.75,1.915,1.91,1.92
59,wbgt,liljegren-c,thermofeel,2541112557,-0.110902,0.296642,0.544648,0.209145,0.555825,-50.827901,...,0.80,0.845,0.84,0.85,2.125,2.10,2.15,2.425,2.40,2.45
60,wbgt,meteor,pywbgt,2541112557,-0.253002,0.239785,0.489678,0.349316,0.551176,-8.216251,...,1.22,1.175,1.17,1.18,1.855,1.85,1.86,1.815,1.81,1.82
61,wbgt,meteor,thermofeel,2541112557,-0.389234,0.659854,0.812314,0.494589,0.900754,-56.393389,...,1.83,1.845,1.84,1.85,3.075,3.05,3.10,3.225,3.20,3.25


## Annual agreement matrices

In [6]:
statistic_specs = {
    "n": {"label": "Matched observations", "unit": "rows"},
    "mean_difference_c": {"label": "Mean difference", "unit": "°C", "signed": True, "reverse_sign": True},
    "stddev_difference_c": {"label": "Difference SD", "unit": "°C"},
    "mean_absolute_difference_c": {"label": "MAE", "unit": "°C"},
    "rmse_c": {"label": "RMSE", "unit": "°C"},
    "loa_lower_c": {"label": "Lower parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_statistic": "loa_upper_c"},
    "loa_upper_c": {"label": "Upper parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_statistic": "loa_lower_c"},
    "median_absolute_difference_c": {"label": "Approximate P50 |difference|", "unit": "°C"},
    "p95_absolute_difference_c": {"label": "Approximate P95 |difference|", "unit": "°C"},
    "p99_absolute_difference_c": {"label": "Approximate P99 |difference|", "unit": "°C"},
    "max_absolute_difference_c": {"label": "Maximum |difference|", "unit": "°C"},
    "area_weighted_mean_difference_c": {"label": "Area-weighted mean difference", "unit": "°C", "signed": True, "reverse_sign": True},
    "area_weighted_stddev_difference_c": {"label": "Area-weighted difference SD", "unit": "°C"},
    "area_weighted_mean_absolute_difference_c": {"label": "Area-weighted MAE", "unit": "°C"},
    "area_weighted_rmse_c": {"label": "Area-weighted RMSE", "unit": "°C"},
    "area_weighted_loa_lower_c": {"label": "Area-weighted lower parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_statistic": "area_weighted_loa_upper_c"},
    "area_weighted_loa_upper_c": {"label": "Area-weighted upper parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_statistic": "area_weighted_loa_lower_c"},
    "area_weighted_median_absolute_difference_c": {"label": "Area-weighted approximate P50 |difference|", "unit": "°C"},
    "area_weighted_p95_absolute_difference_c": {"label": "Area-weighted approximate P95 |difference|", "unit": "°C"},
    "area_weighted_p99_absolute_difference_c": {"label": "Area-weighted approximate P99 |difference|", "unit": "°C"},
}

implementations = sorted(set(yearly_pairwise["implementation_a"]) | set(yearly_pairwise["implementation_b"]))
quantity_matrices = {}

for quantity in quantities:
    statistic_matrices = {
        statistic: pd.DataFrame(np.nan, index=implementations, columns=implementations)
        for statistic in statistic_specs
    }
    for row in yearly_pairwise.loc[yearly_pairwise["quantity"].eq(quantity)].itertuples(index=False):
        for statistic, spec in statistic_specs.items():
            value = getattr(row, statistic)
            statistic_matrices[statistic].loc[row.implementation_a, row.implementation_b] = value
            reverse_value = getattr(row, spec.get("reverse_statistic", statistic))
            statistic_matrices[statistic].loc[row.implementation_b, row.implementation_a] = -reverse_value if spec.get("reverse_sign") else reverse_value
    quantity_matrices[quantity] = statistic_matrices

In [ ]:
quantity_labels = {"wbgt": "WBGT", "tg": "Tg", "tnwb": "Tnwb"}
figures = {}
statistics = list(statistic_specs)
default_statistic = "mean_difference_c"
default_position = statistics.index(default_statistic)

for quantity in quantities:
    statistic_matrices = quantity_matrices[quantity]
    statistic_domains = {}
    for statistic in statistics:
        counterpart = statistic.removeprefix("area_weighted_") if statistic.startswith("area_weighted_") else f"area_weighted_{statistic}"
        domain_statistics = (statistic, counterpart) if counterpart in statistic_matrices else (statistic,)
        finite = np.concatenate([
            values[np.isfinite(values)]
            for domain_statistic in domain_statistics
            for values in [statistic_matrices[domain_statistic].to_numpy()]
        ])
        minimum, maximum = float(np.min(finite)), float(np.max(finite))
        if minimum == maximum:
            padding = max(abs(minimum) * 1e-12, np.finfo(float).eps)
            minimum, maximum = minimum - padding, maximum + padding
        statistic_domains[statistic] = (minimum, maximum)

    figure = go.Figure()
    for position, statistic in enumerate(statistics):
        spec = statistic_specs[statistic]
        matrix = statistic_matrices[statistic]
        minimum, maximum = statistic_domains[statistic]
        signed = spec.get("signed", False)
        figure.add_trace(go.Heatmap(
            z=matrix.to_numpy(), x=implementations, y=implementations,
            text=matrix.map(lambda value: "" if pd.isna(value) else f"{value:,.3g}").to_numpy(),
            texttemplate="%{text}", colorscale="RdBu_r" if signed else "YlOrRd",
            zmin=minimum, zmax=maximum, zmid=0 if signed else None,
            colorbar={"title": {"text": spec["unit"]}}, visible=position == default_position,
            hovertemplate="<b>%{y} − %{x}</b><br>" + spec["label"] + ": %{z:,.5g} " + spec["unit"] + "<extra></extra>",
        ))

    figure.update_layout(
        template="plotly_dark", title=f"{quantity_labels[quantity]}",
        height=760, margin={"l": 120, "r": 80, "t": 180, "b": 100},
        updatemenus=[{
            "buttons": [{"label": statistic_specs[s]["label"], "method": "update", "args": [{"visible": [candidate == s for candidate in statistics]}]} for s in statistics],
            "direction": "down", "showactive": True, "active": default_position,
            "x": 0, "xanchor": "left", "y": 1.08, "yanchor": "bottom",
        }],
        annotations=[{"text": "Statistic", "showarrow": False, "x": 0, "xref": "paper", "xanchor": "left", "y": 1.18, "yref": "paper", "yanchor": "bottom"}],
        xaxis={"title": "implementation_b"},
        yaxis={"title": "implementation_a", "autorange": "reversed", "scaleanchor": "x", "scaleratio": 1},
    )
    figures[quantity] = figure
    display(figure)